In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from helpers import (
    load_data,
    clean_target,
    drop_unused_columns,
    prepare_X_y,
    build_preprocessor
)

In [ ]:
df = load_data("../data/ridings.csv")
df = clean_target(df)
df_model = drop_unused_columns(df)

X, y = prepare_X_y(df_model)

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
preprocessor = build_preprocessor()

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    baseline_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

print("Baseline CV scores:", cv_scores)
print("Mean CV accuracy:", cv_scores.mean())

In [ ]:
baseline_pipeline.fit(X_train, y_train)
y_pred = baseline_pipeline.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
param_grid_lr = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}

grid_lr = GridSearchCV(
    return_train_score=True
    Pipeline(steps=[
        ("preprocessor", build_preprocessor()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    param_grid=param_grid_lr,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

print("Best LR params:", grid_lr.best_params_)
print("Best LR CV score:", grid_lr.best_score_)

In [ ]:
best_lr = grid_lr.best_estimator_
y_pred_best_lr = best_lr.predict(X_test)

print("Best LR Test Accuracy:", accuracy_score(y_test, y_pred_best_lr))
print(classification_report(y_test, y_pred_best_lr))

In [ ]:
param_grid_svm = {
    "model__C": [0.1, 1, 10],
    "model__kernel": ["linear", "rbf", "poly"],
    "model__gamma": ["scale", "auto"]
}

grid_svm = GridSearchCV(
    Pipeline(steps=[
        ("preprocessor", build_preprocessor()),
        ("model", SVC())
    ]),
    param_grid=param_grid_svm,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_svm.fit(X_train, y_train)

print("Best SVM params:", grid_svm.best_params_)
print("Best SVM CV score:", grid_svm.best_score_)

In [ ]:
best_svm = grid_svm.best_estimator_
y_pred_best_svm = best_svm.predict(X_test)

print("Best SVM Test Accuracy:", accuracy_score(y_test, y_pred_best_svm))
print(classification_report(y_test, y_pred_best_svm))
print(confusion_matrix(y_test, y_pred_best_svm))

In [ ]:
results_summary = pd.DataFrame({
    "Model": ["Baseline Logistic Regression", "Tuned Logistic Regression", "Tuned SVM"],
    "CV/Test Score": [
        cv_scores.mean(),
        grid_lr.best_score_,
        grid_svm.best_score_
    ]
})

results_summary

## CV Tuning Summary

- Logistic Regression was used as the baseline model.
- Cross-validation was performed using StratifiedKFold.
- Hyperparameters were tuned using GridSearchCV.
- SVM was also evaluated and compared against Logistic Regression.
- The best model was selected based on cross-validation performance and test accuracy.